# ShadowFox Machine Learning Engineer Internship
## Week 1 — Task 1: Image Tagging with TensorFlow
### Systematic Hyperparameter Tuning & Model Fine-Tuning Experiment

---

### 🎯 Purpose & Scope
This Google Colab notebook conducts a **rigorous, reproducible hyperparameter tuning and fine-tuning experiment** on the **Cats vs. Dogs** computer vision task using transfer learning with **MobileNetV2**.

As identified in our engineering audit:
- The existing production model (`shadowfox_cats_dogs_mobilenetv2.keras`) achieved 98.52% test accuracy using a **completely frozen** feature-extraction backbone with default Adam hyperparameters ($lr=0.001$).
- To complete the internship requirement (*"Optimize model performance with fine-tuning"*), this notebook benchmarks **3 distinct configurations** across learning rates, unfrozen layer depths, head regularizations, and fine-tuning strategies.

### 📋 Experiment Matrix
1. **Config A (Baseline - Feature Extraction):** Completely frozen MobileNetV2 backbone (0 unfrozen layers), Adam ($lr=10^{-3}$), Dropout(0.2) + Dense(1).
2. **Config B (Shallow Fine-Tuning):** 2-stage training; unfreeze top 20 layers of MobileNetV2, Adam ($lr=10^{-5}$), Dropout(0.3) + Dense(1).
3. **Config C (Deep Fine-Tuning):** 2-stage training; unfreeze top 54 layers of MobileNetV2, Adam ($lr=2\times 10^{-5}$), Dropout(0.4) + Dense(32) + Dense(1).

---
### ⚙️ How to Run in Google Colab:
1. Click **Runtime** > **Change runtime type** > Select **T4 GPU** (Hardware Accelerator) > **Save**.
2. Click **Runtime** > **Run all** (or execute cells sequentially using Shift+Enter).
3. At the end of the notebook, `experiment_results.csv` and the best fine-tuned model `shadowfox_cats_dogs_mobilenetv2_finetuned_best.keras` will be saved and ready for download.


## Step 1: Verify Hardware Accelerator (GPU) & Software Environment


In [ ]:
# Verify GPU availability
import os
import tensorflow as tf
import keras

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n[SUCCESS] GPU detected: {gpus[0].name}")
    !nvidia-smi --query-gpu=gpu_name,memory.total,memory.free --format=csv
else:
    print("\n[WARNING] No GPU detected. Please go to Runtime > Change runtime type > T4 GPU for faster training.")


## Step 2: Download & Extract Official Cats vs. Dogs Dataset

We use the official Google MLCC benchmark dataset `cats_and_dogs_filtered.zip` (curated subset of 3,000 images divided into balanced training and validation splits). This dataset matches the standard MobileNetV2 transfer learning benchmark.


In [ ]:
import os
import pathlib
import zipfile
import urllib.request
import tensorflow as tf

# Alternate official Google MLCC dataset URL
dataset_url = "https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip"

# Download target directory
base_cache_dir = pathlib.Path(os.path.expanduser("~/.keras/datasets"))
base_cache_dir.mkdir(parents=True, exist_ok=True)
zip_path = base_cache_dir / "cats_and_dogs_filtered.zip"
extract_dir = base_cache_dir

# 1. Download dataset with User-Agent header (prevent HTTP 403 Forbidden)
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    print(f"Downloading dataset from: {dataset_url}")
    try:
        zip_file_str = tf.keras.utils.get_file(
            "cats_and_dogs_filtered.zip",
            origin=dataset_url,
            extract=False
        )
        zip_path = pathlib.Path(zip_file_str)
    except Exception as dl_err:
        print(f"tf.keras.utils.get_file notice: {dl_err}. Downloading with urllib...")
        req = urllib.request.Request(dataset_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(zip_path, "wb") as out_file:
            total_size = int(resp.headers.get("Content-Length", 0))
            downloaded = 0
            chunk_size = 1024 * 1024
            while True:
                chunk = resp.read(chunk_size)
                if not chunk:
                    break
                out_file.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    pct = (downloaded / total_size) * 100
                    print(f"\rProgress: {downloaded / (1024*1024):.1f} MB / {total_size / (1024*1024):.1f} MB ({pct:.1f}%)", end="")
            print("\nDownload completed successfully.")

# 2. Check that downloaded ZIP is valid before extraction
print(f"\nValidating ZIP archive integrity: {zip_path}")
if not zipfile.is_zipfile(zip_path):
    raise ValueError(f"Downloaded file at {zip_path} is corrupted or not a valid ZIP archive.")

with zipfile.ZipFile(zip_path, "r") as zf:
    test_result = zf.testzip()
    if test_result is not None:
        raise ValueError(f"Corrupted file detected inside ZIP: {test_result}. Please re-download.")
    print(f"[VERIFIED] ZIP archive is valid and intact ({round(zip_path.stat().st_size / (1024*1024), 2)} MB).")
    print(f"Extracting to: {extract_dir}...")
    zf.extractall(extract_dir)

# 3. Preserve existing directory structure
dataset_dir = extract_dir / "cats_and_dogs_filtered"
train_dir = dataset_dir / "train"
val_dir = dataset_dir / "validation"

# 4. Verify that all four directories exist and contain images
subdirs = {
    "train/cats": train_dir / "cats",
    "train/dogs": train_dir / "dogs",
    "validation/cats": val_dir / "cats",
    "validation/dogs": val_dir / "dogs"
}

print(f"\nVerifying dataset directories at {dataset_dir}:")
image_counts = {}
for name, dir_path in subdirs.items():
    assert dir_path.exists(), f"Required directory '{name}' does not exist at {dir_path}!"
    images = list(dir_path.glob("*.jpg")) + list(dir_path.glob("*.jpeg"))
    count = len(images)
    assert count > 0, f"Directory '{name}' exists at {dir_path} but contains 0 images!"
    image_counts[name] = count
    print(f"  [OK] {name:<18} -> {count} images verified")

total_train = image_counts["train/cats"] + image_counts["train/dogs"]
total_val = image_counts["validation/cats"] + image_counts["validation/dogs"]

print(f"\nDataset Summary:")
print(f"  Total Training Images:   {total_train} (Cats: {image_counts['train/cats']}, Dogs: {image_counts['train/dogs']})")
print(f"  Total Validation Images: {total_val} (Cats: {image_counts['validation/cats']}, Dogs: {image_counts['validation/dogs']})")
print(f"  Total Images:            {total_train + total_val}")
print("[SUCCESS] All four directories verified with complete image sets!")


## Step 3: Build Efficient Data Loading Pipeline

Images are loaded at $160 \times 160$ pixels (matching MobileNetV2 input dimensions) in batches of 32 with prefetching enabled for maximum GPU throughput.


In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (160, 160)
SEED = 42

# Create training and validation datasets with deterministic seed for fair comparison
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    seed=SEED
)

# Apply memory caching and asynchronous prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

class_names = ["Cat", "Dog"]
print("\nClass names:", class_names)


## Step 4: Shared Architecture Components & Evaluation Helper

All configurations share the same data augmentation and MobileNetV2 preprocessing pipeline to ensure a strictly controlled, scientific comparison.


In [ ]:
import time
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Data Augmentation pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Results store
experiment_logs = []

def evaluate_and_log(model, val_ds, config_name, strategy, lr_desc, unfrozen_count, head_desc, train_time):
    """
    Evaluate trained model on validation set with robust multi-layer metric extraction.
    Supports Keras 2, Keras 3, and TensorFlow 2.x without metrics_names truncation bugs.
    """
    # 1. Primary evaluation via return_dict=True (standard in modern Keras/TF)
    eval_dict = model.evaluate(val_ds, verbose=0, return_dict=True)
    
    val_loss = eval_dict.get("loss", None)
    val_acc = eval_dict.get("accuracy", eval_dict.get("binary_accuracy", None))
    val_prec = eval_dict.get("precision", None)
    val_rec = eval_dict.get("recall", None)
    
    # 2. Defensive verification: If any metric returned None or 0.0, compute directly from model.predict
    if val_acc is None or val_prec is None or val_rec is None or val_acc == 0.0:
        print(f"Notice: Running direct ground-truth validation tensor evaluation for {config_name}...")
        y_true_list = []
        y_pred_probs = []
        for images, labels in val_ds:
            preds = model(images, training=False)
            probs = tf.sigmoid(preds).numpy().ravel()
            y_true_list.extend(labels.numpy().ravel())
            y_pred_probs.extend(probs)
            
        y_true = np.array(y_true_list)
        y_probs = np.array(y_pred_probs)
        y_pred = (y_probs >= 0.5).astype(int)
        
        val_acc = np.mean(y_true == y_pred)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        val_prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        val_rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        if val_loss is None:
            eps = 1e-7
            y_probs_clipped = np.clip(y_probs, eps, 1 - eps)
            val_loss = -np.mean(y_true * np.log(y_probs_clipped) + (1 - y_true) * np.log(1 - y_probs_clipped))
            
    # Compute harmonic mean (F1 Score)
    f1 = 2 * (val_prec * val_rec) / (val_prec + val_rec + 1e-7)
    
    record = {
        "Configuration": config_name,
        "Strategy": strategy,
        "Learning_Rate": lr_desc,
        "Unfrozen_Layers": unfrozen_count,
        "Head_Architecture": head_desc,
        "Val_Loss": round(float(val_loss), 4),
        "Val_Accuracy": round(float(val_acc) * 100, 2),
        "Val_Precision": round(float(val_prec) * 100, 2),
        "Val_Recall": round(float(val_rec) * 100, 2),
        "Val_F1_Score": round(float(f1) * 100, 2),
        "Train_Time_Sec": round(float(train_time), 1)
    }
    experiment_logs.append(record)
    
    print(f"\n{'='*60}")
    print(f"RESULTS FOR {config_name}:")
    print(f"  Val Accuracy:  {record['Val_Accuracy']}%")
    print(f"  Val Loss:      {record['Val_Loss']}")
    print(f"  Val Precision: {record['Val_Precision']}%")
    print(f"  Val Recall:    {record['Val_Recall']}%")
    print(f"  Val F1-Score:  {record['Val_F1_Score']}%")
    print(f"  Training Time: {record['Train_Time_Sec']}s")
    print(f"{'='*60}\n")
    return record


## Step 5: Configuration A (Baseline — Frozen Feature Extraction)

- **Strategy:** Pure feature extraction (same strategy as the existing production model).
- **Base Model:** MobileNetV2 with all 154 layers frozen (0 unfrozen).
- **Head:** `Dropout(0.2)` + `Dense(1)`
- **Optimizer:** `Adam(learning_rate=0.001)`
- **Regularization:** EarlyStopping (`patience=3`, `restore_best_weights=True`)


In [ ]:
print("\n--- Running Configuration A: Baseline (Frozen Backbone) ---")

# 1. Base model
base_model_a = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
base_model_a.trainable = False

# 2. Build full model
inputs_a = tf.keras.Input(shape=(160, 160, 3))
x_a = data_augmentation(inputs_a)
x_a = layers.Lambda(preprocess_input, name="preprocess_input")(x_a)
x_a = base_model_a(x_a, training=False)
x_a = layers.GlobalAveragePooling2D()(x_a)
x_a = layers.Dropout(0.2)(x_a)
outputs_a = layers.Dense(1, activation=None)(x_a)
model_a = tf.keras.Model(inputs_a, outputs_a, name="config_a_baseline")

# 3. Compile
model_a.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        tf.keras.metrics.Precision(thresholds=0.0, name="precision"),
        tf.keras.metrics.Recall(thresholds=0.0, name="recall")
    ]
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

start_time = time.time()
history_a = model_a.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)
train_time_a = time.time() - start_time

evaluate_and_log(
    model=model_a,
    val_ds=val_dataset,
    config_name="Config A (Baseline)",
    strategy="Feature Extraction (Frozen)",
    lr_desc="1e-3",
    unfrozen_count=0,
    head_desc="Dropout(0.2) + Dense(1)",
    train_time=train_time_a
)


## Step 6: Configuration B (Shallow Fine-Tuning — Top 20 Layers)

- **Strategy:** 2-Stage Fine-Tuning.
  - *Stage 1:* Warm up the classification head with frozen backbone (5 epochs, `lr=1e-3`).
  - *Stage 2:* Unfreeze the top 20 layers of MobileNetV2 (`layers[134:]`).
- **Optimizer:** `Adam(learning_rate=1e-5)` (100x lower to preserve core visual representations).
- **Head:** `Dropout(0.3)` + `Dense(1)`
- **Regularization:** EarlyStopping (`patience=3`, `restore_best_weights=True`)


In [ ]:
print("\n--- Running Configuration B: Shallow Fine-Tuning (Top 20 Layers) ---")

# 1. Base model
base_model_b = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
base_model_b.trainable = False

# 2. Build full model
inputs_b = tf.keras.Input(shape=(160, 160, 3))
x_b = data_augmentation(inputs_b)
x_b = layers.Lambda(preprocess_input, name="preprocess_input")(x_b)
x_b = base_model_b(x_b, training=False)
x_b = layers.GlobalAveragePooling2D()(x_b)
x_b = layers.Dropout(0.3)(x_b)
outputs_b = layers.Dense(1, activation=None)(x_b)
model_b = tf.keras.Model(inputs_b, outputs_b, name="config_b_shallow_tuning")

# Compile Stage 1
model_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        tf.keras.metrics.Precision(thresholds=0.0, name="precision"),
        tf.keras.metrics.Recall(thresholds=0.0, name="recall")
    ]
)

start_time_b = time.time()
print("Stage 1: Warming up classification head (5 epochs)...")
model_b.fit(train_dataset, epochs=5, validation_data=val_dataset, verbose=1)

# Stage 2: Unfreeze top 20 layers
base_model_b.trainable = True
fine_tune_at_b = len(base_model_b.layers) - 20
for layer in base_model_b.layers[:fine_tune_at_b]:
    layer.trainable = False

print(f"Stage 2: Unfreezing from layer {fine_tune_at_b} to {len(base_model_b.layers)} ({len(base_model_b.layers) - fine_tune_at_b} layers)...")

model_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        tf.keras.metrics.Precision(thresholds=0.0, name="precision"),
        tf.keras.metrics.Recall(thresholds=0.0, name="recall")
    ]
)

history_b = model_b.fit(
    train_dataset,
    epochs=15,
    initial_epoch=5,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)
train_time_b = time.time() - start_time_b

evaluate_and_log(
    model=model_b,
    val_ds=val_dataset,
    config_name="Config B (Shallow Fine-Tuning)",
    strategy="2-Stage Fine-Tuning",
    lr_desc="1e-3 -> 1e-5",
    unfrozen_count=20,
    head_desc="Dropout(0.3) + Dense(1)",
    train_time=train_time_b
)


## Step 7: Configuration C (Deep Fine-Tuning — Top 54 Layers + Dense Layer)

- **Strategy:** 2-Stage Fine-Tuning with expanded representation capacity.
  - *Stage 1:* Warm up the classification head (5 epochs, `lr=1e-3`).
  - *Stage 2:* Unfreeze the top 54 layers of MobileNetV2 (from layer 100 onwards).
- **Optimizer:** `Adam(learning_rate=2e-5)`
- **Head:** `Dropout(0.4)` + `Dense(32, activation='relu')` + `Dropout(0.2)` + `Dense(1)`
- **Regularization:** EarlyStopping (`patience=3`, `restore_best_weights=True`)


In [ ]:
print("\n--- Running Configuration C: Deep Fine-Tuning (Top 54 Layers) ---")

# 1. Base model
base_model_c = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
base_model_c.trainable = False

# 2. Build full model with enriched head
inputs_c = tf.keras.Input(shape=(160, 160, 3))
x_c = data_augmentation(inputs_c)
x_c = layers.Lambda(preprocess_input, name="preprocess_input")(x_c)
x_c = base_model_c(x_c, training=False)
x_c = layers.GlobalAveragePooling2D()(x_c)
x_c = layers.Dropout(0.4)(x_c)
x_c = layers.Dense(32, activation="relu")(x_c)
x_c = layers.Dropout(0.2)(x_c)
outputs_c = layers.Dense(1, activation=None)(x_c)
model_c = tf.keras.Model(inputs_c, outputs_c, name="config_c_deep_tuning")

# Compile Stage 1
model_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        tf.keras.metrics.Precision(thresholds=0.0, name="precision"),
        tf.keras.metrics.Recall(thresholds=0.0, name="recall")
    ]
)

start_time_c = time.time()
print("Stage 1: Warming up enriched classification head (5 epochs)...")
model_c.fit(train_dataset, epochs=5, validation_data=val_dataset, verbose=1)

# Stage 2: Unfreeze top 54 layers (from layer 100 onwards)
base_model_c.trainable = True
fine_tune_at_c = 100
for layer in base_model_c.layers[:fine_tune_at_c]:
    layer.trainable = False

print(f"Stage 2: Unfreezing from layer {fine_tune_at_c} to {len(base_model_c.layers)} ({len(base_model_c.layers) - fine_tune_at_c} layers)...")

model_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        tf.keras.metrics.Precision(thresholds=0.0, name="precision"),
        tf.keras.metrics.Recall(thresholds=0.0, name="recall")
    ]
)

history_c = model_c.fit(
    train_dataset,
    epochs=15,
    initial_epoch=5,
    validation_data=val_dataset,
    callbacks=[early_stopping],
    verbose=1
)
train_time_c = time.time() - start_time_c

evaluate_and_log(
    model=model_c,
    val_ds=val_dataset,
    config_name="Config C (Deep Fine-Tuning)",
    strategy="2-Stage Fine-Tuning",
    lr_desc="1e-3 -> 2e-5",
    unfrozen_count=54,
    head_desc="Dropout(0.4) + Dense(32) + Dense(1)",
    train_time=train_time_c
)


## Step 8: Results Comparison Table & Visualizations

We consolidate the measured validation metrics into a single structured comparison table and export it directly to `experiment_results.csv`.


In [ ]:
import matplotlib.pyplot as plt

# 1. Create DataFrame
df_results = pd.DataFrame(experiment_logs)

# 2. Save CSV
csv_filename = "experiment_results.csv"
df_results.to_csv(csv_filename, index=False)
print(f"Saved experiment results to {csv_filename}\n")

# 3. Display Markdown & Pretty Table
try:
    print(df_results.to_markdown(index=False))
except Exception:
    print(df_results)

# 4. Plot Comparison Bar Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(df_results))
width = 0.35

# Plot Accuracy & F1
axes[0].bar(x - width/2, df_results["Val_Accuracy"], width, label="Val Accuracy (%)", color="#4f46e5")
axes[0].bar(x + width/2, df_results["Val_F1_Score"], width, label="Val F1 Score (%)", color="#10b981")
axes[0].set_title("Validation Accuracy & F1 Score", fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_results["Configuration"], rotation=12)
axes[0].set_ylim([90, 100])
axes[0].legend()
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Plot Precision & Recall
axes[1].bar(x - width/2, df_results["Val_Precision"], width, label="Precision (%)", color="#06b6d4")
axes[1].bar(x + width/2, df_results["Val_Recall"], width, label="Recall (%)", color="#f59e0b")
axes[1].set_title("Validation Precision & Recall", fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_results["Configuration"], rotation=12)
axes[1].set_ylim([90, 100])
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

# Plot Validation Loss
axes[2].bar(df_results["Configuration"], df_results["Val_Loss"], color="#ef4444", width=0.45)
axes[2].set_title("Validation Loss (Lower is Better)", fontsize=12, fontweight='bold')
axes[2].set_xticklabels(df_results["Configuration"], rotation=12)
axes[2].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("tuning_comparison_chart.png", dpi=300)
plt.show()


## Step 9: Identify Best Configuration & Export Artifacts

The best configuration is automatically chosen based on highest Validation Accuracy (with Validation Loss as tie-breaker). It is saved as a separate file without modifying the existing production model.


In [ ]:
# Find best model
best_row = df_results.sort_values(by=["Val_Accuracy", "Val_Loss"], ascending=[False, True]).iloc[0]
best_config_name = best_row["Configuration"]

print(f"★ BEST CONFIGURATION: {best_config_name}")
print(f"  - Val Accuracy: {best_row['Val_Accuracy']}%")
print(f"  - Val Loss:     {best_row['Val_Loss']}")
print(f"  - Val F1 Score: {best_row['Val_F1_Score']}%")
print(f"  - Strategy:     {best_row['Strategy']}")

# Map to the corresponding model instance
model_dict = {
    "Config A (Baseline)": model_a,
    "Config B (Shallow Fine-Tuning)": model_b,
    "Config C (Deep Fine-Tuning)": model_c
}

best_model = model_dict[best_config_name]
best_model_path = "shadowfox_cats_dogs_mobilenetv2_finetuned_best.keras"

# Save as separate model file (preserves original shadowfox_cats_dogs_mobilenetv2.keras)
best_model.save(best_model_path)
print(f"\n[SUCCESS] Best fine-tuned model saved as '{best_model_path}' ({round(os.path.getsize(best_model_path)/(1024*1024), 2)} MB)")


## Step 10: Download Artifacts to Local Machine

Run this cell to download `experiment_results.csv`, `tuning_comparison_chart.png`, and the best fine-tuned model artifact directly to your local computer.


In [ ]:
try:
    from google.colab import files
    print("Downloading experiment results CSV...")
    files.download("experiment_results.csv")

    print("Downloading comparison chart...")
    files.download("tuning_comparison_chart.png")

    print("Downloading best fine-tuned model artifact (~9-10 MB)...")
    files.download("shadowfox_cats_dogs_mobilenetv2_finetuned_best.keras")
    print("\nAll deliverables ready for inclusion in your ShadowFox Task 1 report!")
except Exception as e:
    print(f"Local environment detected or download error: {e}")
    print("Files are saved in the current working directory.")
